# DFX Intelligence MCP quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Capital-W-Holdings/us-property-parcel-real-estate-debt/blob/main/examples/quickstart.ipynb)

DFX Intelligence is a remote MCP server (Streamable HTTP) with cited answers from SEC filings and other public records:
which BDCs lend to a company, what each position is marked at, which plans committed to a manager. No account and no
key are needed for anything in this notebook.

Two ways in:

1. **Raw JSON-RPC over httpx**, so you can see exactly what goes over the wire.
2. **The OpenAI Agents SDK**, attaching DFX to an agent with `MCPServerStreamableHttp`.

The outputs saved below are from a real run against production; the time of that run is printed in the last cell.

In [1]:
%pip install -q --disable-pip-version-check "httpx==0.28.1" "openai-agents==0.24.0"

Note: you may need to restart the kernel to use updated packages.


## 1. Raw JSON-RPC

The core profile URL below carries `?c=ex-notebook`, a channel tag that tells DFX the call came from this notebook.
Keep it as is. Every call is a POST of one JSON-RPC message; the server answers with JSON.

In [2]:
import json
import os

import httpx

MCP = "https://exchange-production-9123.up.railway.app/mcp/core?c=ex-notebook"
QUESTION = "Which BDCs lend to Acquia, and what are the positions marked at?"

# DFX_EXAMPLE_UA is only for DFX's own test runs, so they can be excluded from usage counts. Leave it unset.
headers = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}
if os.environ.get("DFX_EXAMPLE_UA"):
    headers["User-Agent"] = os.environ["DFX_EXAMPLE_UA"]
http = httpx.Client(headers=headers, timeout=120)


def rpc(method, params=None, rpc_id=1):
    r = http.post(MCP, json={"jsonrpc": "2.0", "id": rpc_id, "method": method, "params": params or {}})
    r.raise_for_status()
    return r.json()["result"]


init = rpc("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                          "clientInfo": {"name": "dfx-quickstart-notebook", "version": "1.0"}})
print(init["serverInfo"])
tools = rpc("tools/list", rpc_id=2)["tools"]
print(len(tools), "tools:", ", ".join(t["name"] for t in tools[:6]), "...")

{'name': 'dfx-real-estate', 'title': 'DFX Intelligence', 'version': '0.16.0'}


20 tools: research, what_can_dfx_answer, resolve_name, search_entities, get_entity, search_relationships ...


In [3]:
result = rpc("tools/call", {"name": "research", "arguments": {"question": QUESTION}}, rpc_id=3)
answer = result["structuredContent"]          # the same JSON is also in result["content"][0]["text"]

print("state:", answer["state"])
print("answer:", answer["answer"]["text"])
for row in answer["relationships"]:
    print(f"  {row['subject']['name']} | fair value ${row['amount_usd']:,.0f} | marked {row.get('mark_pct')}%"
          f" | as of {row['as_of']} | {row['cite']['url']}")

state: OK
answer: At 2026-06-30, 10 BDCs held positions in Acquia Inc on their own schedules of investments, named holders marked 92 to 99% of principal; the largest: Blue Owl Technology Finance Corp. ($183.1M of debt at fair value over 2 tranches; principal $188.3M), Goldman Sachs BDC, Inc. ($54.6M of debt at fair value over 3 tranches; principal $56.0M), Oaktree Specialty Lending Corp ($33.3M of debt at fair value over 3 tranches; principal $34.4M), Goldman Sachs Private Middle Market Credit II LLC ($26.2M of debt at fair value over 2 tranches; principal $26.9M), and 6 more on the next page (continue with page.continuation).
  Blue Owl Technology Finance Corp. | fair value $183,120,000 | marked 97.3% | as of 2026-06-30 | https://www.sec.gov/Archives/edgar/data/1747777/000174777726000028/ortf-20260630.htm
  Goldman Sachs BDC, Inc. | fair value $54,586,000 | marked 97.5% | as of 2026-06-30 | https://www.sec.gov/Archives/edgar/data/1572694/000119312526338331/gsbd-20260630.htm
  Oaktree 

Every row cites the lender's own filing on sec.gov. `NO_MATCH`, `UNKNOWN` and `NOT_COVERED` are answers
too: they mean "do not conclude no", and `coverage_limits` says why. More pages come back through
`page.continuation`:

In [4]:
page = answer["page"]
print(f"page {page['index']} of {page['pages']}, {page['rows_total']} rows in all")
if page.get("continuation"):
    nxt = rpc("tools/call", {"name": "research", "arguments": {"continue": page["continuation"]}}, rpc_id=4)
    for row in nxt["structuredContent"]["relationships"]:
        print(f"  {row['subject']['name']} | {row['type']} | {row['cite']['url']}")

page 1 of 5, 17 rows in all


  Vista Equity Partners | SPONSOR_OF_BORROWER | https://www.wcapgroup.com/wp-content/uploads/2019/09/Acquia-Announces-Majority-Investment-from-Vista-Equity-Partners.pdf


## 2. The same server in the OpenAI Agents SDK

`MCPServerStreamableHttp` connects to the same URL and hands the agent every DFX tool. Set `OPENAI_API_KEY` and the
model decides when to call `research`; without a key this cell calls `research` on the agent's own MCP connection, so
you see what the model would receive. Jupyter and Colab both allow `await` at the top level of a cell.

In [5]:
from agents import Agent, Runner
from agents.mcp import MCPServerStreamableHttp

params = {"url": MCP, "timeout": 120, "sse_read_timeout": 300,
          "headers": {"User-Agent": os.environ["DFX_EXAMPLE_UA"]} if os.environ.get("DFX_EXAMPLE_UA") else {}}

async with MCPServerStreamableHttp(name="dfx", params=params, client_session_timeout_seconds=120) as dfx:
    agent = Agent(name="credit analyst", mcp_servers=[dfx],
                  instructions="Answer with DFX research and cite the evidence URL for every figure.")
    if os.environ.get("OPENAI_API_KEY"):
        print((await Runner.run(agent, QUESTION)).final_output)
    else:
        res = await dfx.call_tool("research", {"question": "Which BDCs lend to Hyland Software?"})
        a = res.structured_content          # mcp 2.x; the raw JSON-RPC above calls it structuredContent
        print("state:", a["state"])
        print("answer:", a["answer"]["text"])
        for row in a["relationships"][:3]:
            print(f"  {row['subject']['name']} | {row['cite']['url']}")

state: OK
answer: At 2026-06-30, 21 BDCs held positions in Hyland Software, Inc on their own schedules of investments, named holders marked 97 to 100% of principal; the largest: Blue Owl Technology Finance Corp. ($142.9M of debt at fair value over 2 tranches; principal $147.5M), Blue Owl Credit Income Corp. ($139.2M of debt at fair value over 2 tranches; principal $143.6M), ARES CAPITAL CORP ($108.6M of debt at fair value), Goldman Sachs Private Credit Corp. ($92.6M of debt at fair value over 2 tranches; principal $93.1M), and 17 more on the next page (continue with page.continuation). ARES CAPITAL CORP ($226.8M of debt at fair value) reported for 2026-09-30; 20 of the 21 holders at 2026-06-30 have not filed for 2026-09-30 yet, so the holders above are counted at 2026-06-30.
  Blue Owl Technology Finance Corp. | https://www.sec.gov/Archives/edgar/data/1747777/000174777726000028/ortf-20260630.htm
  Blue Owl Credit Income Corp. | https://www.sec.gov/Archives/edgar/data/1812554/0001812554

In [6]:
import datetime
print("ran at", datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"))

ran at 2026-10-29T18:36:47Z


## Next

* Add DFX to your editor: Cursor, VS Code and Claude Code install lines are in
  [docs/CONNECT.md](https://github.com/Capital-W-Holdings/us-property-parcel-real-estate-debt/blob/main/docs/CONNECT.md).
* LangChain, OpenAI Agents SDK and LlamaIndex scripts with recorded runs:
  [examples/mcp/](https://github.com/Capital-W-Holdings/us-property-parcel-real-estate-debt/tree/main/examples/mcp).
* Anonymous research is metered per network address; a refused call costs nothing and says when to retry. Members get
  more through a `dfx_live_` key or OAuth sign-in: https://dfxintel.com/agents